In [0]:
# ---------------------------------------------------------
# 0. Run Imports 
# ---------------------------------------------------------

import os
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
import hashlib


In [0]:
# ---------------------------------------------------------
# 1. Set-up Ingestion Audit Log
# ---------------------------------------------------------

def log_ingestion_audit(
    spark,
    audit_table,
    source_system,
    bronze_table,
    file_path,
    batch_id,
    pipeline_run_id,
    start_time,
    end_time,
    row_count,
    file_size,
    status
):

    duration_seconds = (end_time - start_time)

    df = spark.createDataFrame(
        [(
            source_system,
            bronze_table,
            file_path,
            batch_id,
            pipeline_run_id,
            row_count,
            file_size,
            duration_seconds,
            status
        )],
        [
            "source_system",
            "bronze_table",
            "file_path",
            "batch_id",
            "pipeline_run_id",
            "row_count",
            "file_size",
            "ingestion_duration_seconds",
            "status"
        ]
    ).withColumn(
        "start_time",
        F.lit(start_time)
    ).withColumn(
        "end_time",
        F.lit(end_time)
    ).withColumn(
        "audit_timestamp",
        F.current_timestamp()
    )

    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(audit_table)
    )